# Price-point analysis

The BigQuery SQL returns analysis-ready subscription-level data. `summarize_price_points` groups that data by any requested dimensions and calculates user counts, contact rate, churn rate, save rate, gamer rate, revenue, and CNRC.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import bigquery
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.price_point_helpers import plot_metric, summarize_price_points

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Load detail data

In [ ]:
project_id = "gannett-datascience"
sql_path = PROJECT_ROOT / "src" / "sql" / "price_point_detail_data.sql"
client = bigquery.Client(project=project_id)
df = client.query(sql_path.read_text()).to_dataframe(
    create_bqstorage_client=False
)

print(
    f"Loaded {len(df):,} mature subscriptions across "
    f"{df['market'].nunique(dropna=True):,} markets."
)

## 2. Tables

Pass any columns to `dimensions`. Subset `df` with normal pandas code when an analysis needs a specific population.

In [ ]:
overall = summarize_price_points(df)
display(overall.round(3))

cohort_treatment = summarize_price_points(df, ["cohort", "treatment"])
display(cohort_treatment.round(3))

three_offer = df.loc[df["cohort"].eq("Three-Offer Cohort")]
price_treatment = summarize_price_points(
    three_offer, ["start_price_group", "treatment"]
)
display(price_treatment.round(3))

contacted = df.loc[df["contacted"].eq(1)]
gamer_treatment = summarize_price_points(
    contacted, ["gamer_type", "treatment"]
)
display(
    gamer_treatment[[
        "gamer_type", "treatment", "users",
        "churn_rate", "total_revenue", "cnrc",
    ]].round(3)
)

market_summary = (
    summarize_price_points(df, ["market"])
    .loc[lambda result: result["users"].ge(200)]
    .sort_values("contact_rate", ascending=False)
)
display(market_summary.head(30).round(3))

## 3. Charts

In [ ]:
plot_metric(
    cohort_treatment,
    metric="contact_rate",
    x="cohort",
    hue="treatment",
    title="Contact rate by cohort and treatment",
)
plt.show()

plot_metric(
    price_treatment,
    metric="churn_rate",
    x="start_price_group",
    hue="treatment",
    title="Three-Offer churn rate by start-price group and treatment",
)
plt.show()

plot_metric(
    gamer_treatment,
    metric="cnrc",
    x="gamer_type",
    hue="treatment",
    title="CNRC by gamer type and treatment",
)
plt.show()